In [ ]:
#load data
import pandas as pd

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))
from dotenv import load_dotenv
import os

#from feature_engineering.feature_engineering_experimentation_pipeline import build_ts_players
from helldivers2.feature_engineering.feature_engineering_experimentation_pipeline import build_ts_players
ts_players = build_ts_players()
ts_players.head()

In [ ]:
ts_players.shape

In [ ]:
#show columns in df


print(ts_players.columns)
pd.set_option('display.max_columns', None)
print(ts_players.head())

In [ ]:
import numpy as np
#if you want to add more FEATURE DO IT HERE THEN RUN THE BELOW BLOCK
ts_players["delta_1"] = ts_players["total_players"].diff(1)
ts_players["delta_2"] = ts_players["total_players"].diff(2)

for lag in range(1, 4):
    ts_players[f"lag_{lag}"] = ts_players["total_players"].shift(lag)
    
# ts_players["rolling_mean_6"] = ts_players["total_players"].rolling(6).mean()
# ts_players["rolling_std_6"] = ts_players["total_players"].rolling(6).std()

ts_players["rolling_mean_6"] = ts_players["total_players"].shift(1).rolling(6).mean()
ts_players["rolling_std_6"] = ts_players["total_players"].shift(1).rolling(6).std()

ts_players["delta_1"] = ts_players["total_players"] - ts_players["lag_1"]
ts_players["delta_3"] = ts_players["total_players"] - ts_players["lag_3"]

ts_players["lag_24"] = ts_players["total_players"].shift(24)

#just added below
ts_players["lag_168"] = ts_players["total_players"].shift(168)  # 24*7
ts_players["rolling_mean_24"] = ts_players["total_players"].shift(1).rolling(24).mean()
ts_players["rolling_std_24"] = ts_players["total_players"].shift(1).rolling(24).std()

# weekly_avg = ts_players.groupby("hour")["total_players"].mean()
# ts_players["weekly_expected"] = ts_players["hour"].map(weekly_avg)


#added below just now

ts_players["is_dropping_fast"] = (ts_players["delta_1"] < -5000).astype(int)
ts_players["acceleration"] = ts_players["delta_1"].diff()


ts_players["target"] = ts_players["total_players"].shift(-1) - ts_players["total_players"]


ts_players["change_1"] = ts_players["total_players"] - ts_players["lag_1"]
ts_players["change_2"] = ts_players["lag_1"] - ts_players["lag_2"]
ts_players["trend_strength"] = ts_players["change_1"] - ts_players["change_2"]
#end of just added
ts_players = ts_players.dropna()
ts_players = ts_players.drop(columns=["mo_players"])



#just added below
# ts_players["velocity_1"] = ts_players["total_players"].diff(1)
# ts_players["velocity_6"] = ts_players["total_players"].diff(6)
# ts_players["velocity_12"] = ts_players["total_players"].diff(12)

# ts_players["lag1_vs_24"] = ts_players["lag_1"] - ts_players["lag_24"]

# THEN:

# --- TARGET ---
ts_players["target"] = ts_players["total_players"].shift(-1)

# --- LOG TARGET ---
ts_players["target_log"] = np.log1p(ts_players["target"])

# --- FINAL CLEAN (CRITICAL) ---
ts_players = ts_players.replace([np.inf, -np.inf], np.nan)
ts_players = ts_players.dropna().reset_index(drop=True)

# --- SPLIT ---
X = ts_players.drop(columns=["total_players", "timestamp", "player_on_planet", "target", "target_log"])
y_log = ts_players["target_log"]
y_actual = ts_players["target"]



# #drop na's
# ts_players = ts_players.dropna()
# #splitting data
from sklearn.model_selection import TimeSeriesSplit
# #turn into X and y's
# X = ts_players.drop(columns=["total_players", "timestamp", "player_on_planet"])
# y = ts_players["total_players"]
# y = ts_players["total_players"].shift(-1)
# y_log = np.log1p(y)
# #drop na's

tscv = TimeSeriesSplit(n_splits=5)

# for train_index, test_index in tscv.split(X):
#     X_train, X_test = X.iloc[train_index], X.iloc[test_index]
#     y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# create ALL features first



In [ ]:
#add lags
ts_players["lag_6"] = ts_players["total_players"].shift(6)
ts_players["lag_12"] = ts_players["total_players"].shift(12)
ts_players["lag_24"] = ts_players["total_players"].shift(24)


#below is what the pros do so lets try it out
df = ts_players.copy()

# level
df["target_level"] = df["total_players"].shift(-1)

# delta
df["target_delta"] = df["total_players"].shift(-1) - df["total_players"]

# log-delta
df["log_players"] = np.log1p(df["total_players"])
df["target_log_delta"] = df["log_players"].shift(-1) - df["log_players"]

# clean
df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)


features = [
    "lag_1","lag_2","lag_3",
    "lag_6","lag_12","lag_24","lag_168",
    "hour_sin","hour_cos"
]

X = df[features]

from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

tscv = TimeSeriesSplit(n_splits=5)

results = {
    "level": [],
    "delta": [],
    "log_delta": []
}

for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

    base_train = df["total_players"].iloc[train_idx]
    base_test = df["total_players"].iloc[test_idx]

    model = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=7,
        subsample=0.9,
        colsample_bytree=0.9,
        early_stopping_rounds=50,
        random_state=42
    )

    # -------- LEVEL --------
    y_train = df["target_level"].iloc[train_idx]
    y_test = df["target_level"].iloc[test_idx]

    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    results["level"].append(rmse)

    # -------- DELTA --------
    y_train = df["target_delta"].iloc[train_idx]
    y_test = df["target_delta"].iloc[test_idx]

    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    pred_delta = model.predict(X_test)

    pred_actual = base_test.values + pred_delta

    rmse = np.sqrt(mean_squared_error(df["target_level"].iloc[test_idx], pred_actual))
    results["delta"].append(rmse)

    # -------- LOG DELTA --------
    y_train = df["target_log_delta"].iloc[train_idx]
    y_test = df["target_log_delta"].iloc[test_idx]

    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    pred_log_delta = model.predict(X_test)

    pred_log = np.log1p(base_test.values) + pred_log_delta
    pred_actual = np.expm1(pred_log)

    rmse = np.sqrt(mean_squared_error(df["target_level"].iloc[test_idx], pred_actual))
    results["log_delta"].append(rmse)
    
for k, v in results.items():
    print(k, "AVG RMSE:", np.mean(v))

In [ ]:
#below is what should be best model
model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

X_full = df[features]
y_full = df["target_delta"]

model.fit(X_full, y_full)

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_squared_log_error
from xgboost import XGBRegressor
import joblib

# =========================
# 1. COPY DATA
# =========================
df = ts_players.copy()

# =========================
# 2. FEATURE ENGINEERING
# =========================

# lags
for lag in [1, 2, 3, 6, 12, 24, 168]:
    df[f"lag_{lag}"] = df["total_players"].shift(lag)

# time features (VERY IMPORTANT)
df["hour"] = df["timestamp"].dt.hour
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# strong feature
df["same_hour_yesterday"] = df["lag_24"]

# =========================
# 3. TARGETS (DELTA MODEL)
# =========================
df["target_level"] = df["total_players"].shift(-1)
df["target_delta"] = df["target_level"] - df["total_players"]

# =========================
# 4. CLEAN DATA
# =========================
df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

# =========================
# 5. FEATURES / TARGET
# =========================
features = [
    "lag_1","lag_2","lag_3",
    "lag_6","lag_12","lag_24",
    "hour_sin","hour_cos",
    "same_hour_yesterday"
]

X = df[features]
y = df["target_delta"]

# =========================
# 6. TRAIN / TEST SPLIT (HOLDOUT)
# =========================
split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test_delta = y.iloc[split_idx:]

base_test = df["total_players"].iloc[split_idx:]
y_test_actual = df["target_level"].iloc[split_idx:]

# =========================
# 7. TRAIN FINAL MODEL
# =========================
model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

model.fit(X_train, y_train)

# =========================
# 8. PREDICTIONS
# =========================
pred_delta = model.predict(X_test)

# reconstruct actual prediction
pred_actual = base_test.values + pred_delta

# =========================
# 9. EVALUATION
# =========================
rmse = np.sqrt(mean_squared_error(y_test_actual, pred_actual))
rmsle = np.sqrt(mean_squared_log_error(y_test_actual, pred_actual))

print("FINAL HOLDOUT RMSE:", rmse)
print("FINAL HOLDOUT RMSLE:", rmsle)

# =========================
# 10. PLOT
# =========================
plt.figure(figsize=(12,5))
plt.plot(y_test_actual.values, label="Actual")
plt.plot(pred_actual, label="Predicted")
plt.legend()
plt.title("Final Model Performance (Delta Model)")
plt.show()


# =========================
# BETTER VISUAL EVALUATION
# =========================

# convert to arrays
actual = y_test_actual.values
pred = pred_actual

# ---- 1. LINE PLOT (what you already had, but cleaner) ----
plt.figure(figsize=(14,5))
plt.plot(actual, label="Actual", linewidth=2)
plt.plot(pred, label="Predicted", linestyle="--")
plt.legend()
plt.title("Actual vs Predicted (Holdout)")
plt.xlabel("Time Index")
plt.ylabel("Players")
plt.show()


# ---- 2. ERROR OVER TIME (VERY IMPORTANT) ----
errors = pred - actual

plt.figure(figsize=(14,4))
plt.plot(errors, label="Prediction Error")
plt.axhline(0)
plt.title("Prediction Error Over Time")
plt.xlabel("Time Index")
plt.ylabel("Error (Pred - Actual)")
plt.legend()
plt.show()


# ---- 3. ACTUAL vs PRED SCATTER (MODEL QUALITY) ----
plt.figure(figsize=(6,6))
plt.scatter(actual, pred, alpha=0.6)

# perfect prediction line
min_val = min(actual.min(), pred.min())
max_val = max(actual.max(), pred.max())
plt.plot([min_val, max_val], [min_val, max_val])

plt.title("Actual vs Predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()


# ---- 4. RESIDUAL DISTRIBUTION ----
plt.figure(figsize=(6,4))
plt.hist(errors, bins=30)
plt.title("Error Distribution")
plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.show()


# ---- 5. % ERROR (SUPER USEFUL FOR YOU) ----
pct_error = errors / actual

plt.figure(figsize=(14,4))
plt.plot(pct_error)
plt.axhline(0)
plt.title("Percentage Error Over Time")
plt.xlabel("Time Index")
plt.ylabel("% Error")
plt.show()


# ---- 6. PRINT SUMMARY ----
print("Mean Error:", np.mean(errors))
print("Std Error:", np.std(errors))
print("Max Overprediction:", np.max(errors))
print("Max Underprediction:", np.min(errors))




# =========================
# 11. SAVE MODEL
# =========================
joblib.dump(model, "delta_model.pkl")

print("Model saved as delta_model.pkl")



In [ ]:
from sklearn.metrics import mean_squared_error, mean_squared_log_error, mean_absolute_error
from xgboost import XGBRegressor
import numpy as np

rmsle_scores = []
rmse_scores = []
mae_scores = []

for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y_log.iloc[train_index], y_log.iloc[test_index]

    model = XGBRegressor(
        n_estimators=2500,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=50,
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    # predictions
    y_pred_log = model.predict(X_test)

    # convert back
    y_pred = np.expm1(y_pred_log)
    y_test_actual = np.expm1(y_test)

    # metrics
    mae = mean_absolute_error(y_test_actual, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
    rmsle = np.sqrt(mean_squared_error(y_test, y_pred_log))

    rmsle_scores.append(rmsle)
    rmse_scores.append(rmse)
    mae_scores.append(mae)

# final results
print("AVG RMSLE:", np.mean(rmsle_scores))
print("AVG RMSE:", np.mean(rmse_scores))
print("AVG MAE:", np.mean(mae_scores))

last_y_pred = y_pred
last_y_test = y_test_actual

import matplotlib.pyplot as plt

plt.figure()
plt.plot(last_y_test.values, label="Actual")
plt.plot(last_y_pred, label="Predicted")
plt.legend()
plt.title("Final Fold Prediction")
plt.show()

In [ ]:
#advanced model
#lets try XGboost 
for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y_log.iloc[train_index], y_log.iloc[test_index]


#LETS TRY TRAINING ON Y_LOG INSTEAD OF Y
from sklearn.metrics import mean_squared_error, mean_squared_log_error
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=2500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    early_stopping_rounds=50,
    verbose=True
)
#predict
y_pred_log = model.predict(X_test)
#convert back to original scale
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

mae = mean_absolute_error(y_test_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
rmsle = np.sqrt(mean_squared_log_error(y_test_actual, y_pred))

print("RMSLE:", rmsle)
print("MAE:", mae)
print("RMSE:", rmse)



#evaluate

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

import matplotlib.pyplot as plt

plt.figure()
plt.plot(y_test.values, label="Actual")
plt.plot(y_pred, label="Predicted")
plt.legend()
plt.title("XGBoost: Actual vs Predicted Players")
plt.show()

import pandas as pd

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

# RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# RMSLE
rmsle = np.sqrt(mean_squared_log_error(y_test, y_pred))

print("RMSE:", rmse)
print("RMSLE:", rmsle)

In [ ]:
#advanced model
#lets try XGboost 

from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=2500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    early_stopping_rounds=50,
    verbose=True
)
#predict
y_pred = model.predict(X_test)

#evaluate

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

#mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

import matplotlib.pyplot as plt

plt.figure()
plt.plot(y_test.values, label="Actual")
plt.plot(y_pred, label="Predicted")
plt.legend()
plt.title("XGBoost: Actual vs Predicted Players")
plt.show()

import pandas as pd

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

model = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

results = ts_players.iloc[-len(y_test):].copy()
results["actual"] = y_test.values
results["predicted"] = y_pred

results["abs_error"] = np.abs(results["actual"] - results["predicted"])
results["squared_error"] = (results["actual"] - results["predicted"])**2
results["rmse"] = np.sqrt(results["squared_error"])

In [ ]:
#baseline model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)

#run the test
y_pred = model.predict(X_test)

from sklearn.metrics import mean_absolute_error
#calculate MAE
mae = mean_absolute_error(y_test, y_pred)
print("MAE:", mae)


import matplotlib.pyplot as plt
#plot actual vs predicted
plt.figure()
plt.plot(y_test.values, label="Actual")
plt.plot(y_pred, label="Predicted")
plt.legend()
plt.title("Actual vs Predicted Players")
plt.show()

In [ ]:
ts_players.columns

In [ ]:
#ts_players[["total_players", "mo_players"]].corr()

In [ ]:
#advanced model
#lets try XGboost 

from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=2500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    early_stopping_rounds=50,
    verbose=True
)
#predict
y_pred = model.predict(X_test)

#evaluate

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

import matplotlib.pyplot as plt

plt.figure()
plt.plot(y_test.values, label="Actual")
plt.plot(y_pred, label="Predicted")
plt.legend()
plt.title("XGBoost: Actual vs Predicted Players")
plt.show()

import pandas as pd

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

In [ ]:
#%pip install plotly

In [ ]:
#%pip install ipywidgets

In [ ]:
#lets try bayesian optimization

import optuna
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

def objective(trial):
    params = {
        "n_estimators": 3000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "tree_method": "hist"  # faster
    }

    model = XGBRegressor(**params)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        early_stopping_rounds=50,
        verbose=False
    )

    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    return rmse


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)  # increase to 50–100 for better results

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)

best_params = study.best_params

model = XGBRegressor(
    **best_params,
    n_estimators=3000,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    early_stopping_rounds=50,
    verbose=True
)

y_pred = model.predict(X_test)

plt.figure()
plt.plot(y_test.values, label="Actual")
plt.plot(y_pred, label="Predicted")
plt.legend()
plt.title("Tuned XGBoost: Actual vs Predicted Players")
plt.show()

optuna.visualization.plot_optimization_history(study)

results = ts_players.iloc[-len(y_test):].copy()
results["actual"] = y_test.values
results["predicted"] = y_pred

plt.figure()
plt.plot(results["timestamp"], results["actual"], label="Actual")
plt.plot(results["timestamp"], results["predicted"], label="Predicted")

plt.legend()
plt.xticks(rotation=45)
plt.title("XGBoost: Actual vs Predicted Players")
plt.show()

In [ ]:
#different variations of the model

In [ ]:
#ARIMA CODE WTF THIS IS WEIRD
series = ts_players["total_players"]
series.index = pd.to_datetime(ts_players["timestamp"])

series = series.asfreq("h")
series = series.ffill()   # or .interpolate()

#train ARIMA
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
model = SARIMAX(
    series,
    order=(2,0,2),
    seasonal_order=(1,0,1,24)
)
#model = ARIMA(series, order=(1,1,1))  # (p, d, q)
model_fit = model.fit()

print(model_fit.summary())

#forecast
forecast = model_fit.forecast(steps=24)
print(forecast)

#plot SARIMA
import matplotlib.pyplot as plt

forecast_index = pd.date_range(
    start=series.index[-1],
    periods=25,
    freq="h"
)[1:]

plt.figure(figsize=(12,5))
plt.plot(series[-100:], label="Actual")
plt.plot(forecast_index, forecast, label="Forecast", color="red")
plt.legend()
plt.title("SARIMA Forecast")
plt.show()

In [ ]:
ts_players['timestamp'].max()